<a href="https://colab.research.google.com/github/mandarakn73/nlp/blob/main/exp6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 52.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")


In [ ]:
text = "NLP is very good , intresting subject."
doc = nlp(text)


In [ ]:
tokens = [token.text for token in doc]
print(tokens)


['NLP', ' ', 'is', 'very', 'good', ',', 'intresting', 'subject', '.']


In [ ]:
word_tokens = [token.text for token in doc]
print(word_tokens)


['NLP', 'is', 'very', 'good', ',', 'intresting', 'subject', '.']


In [ ]:
sentence_tokens = [sent.text for sent in doc.sents]
print(sentence_tokens)


['NLP is very good , intresting subject.']


In [ ]:
tokens_no_stopwords = [token.text for token in doc if not token.is_stop and not token.is_punct]
print(tokens_no_stopwords)


['NLP', 'good', 'intresting', 'subject']


In [ ]:
lemmas = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
print(lemmas)


['Apple', 'look', 'buy', 'startup', 'San', 'Francisco', '$', '1', 'billion']


In [ ]:
pos_tags = [(token.text, token.pos_) for token in doc]
pos_tags


[('NLP', 'PROPN'),
 ('is', 'AUX'),
 ('very', 'ADV'),
 ('good', 'ADJ'),
 (',', 'PUNCT'),
 ('intresting', 'VERB'),
 ('subject', 'NOUN'),
 ('.', 'PUNCT')]

In [ ]:
entities = [(ent.text, ent.label_) for ent in doc.ents]
entities


[('NLP', 'ORG')]

In [ ]:
#Conditional Random Field
pip install scikit-learn


In [ ]:
# Example data
train_sents = [
    [("Apple", "ORG"), ("is", "O"), ("buying", "O"), ("a", "O"), ("startup", "O"), ("in", "O"), ("London", "GPE")],
    [("Barack", "PERSON"), ("Obama", "PERSON"), ("was", "O"), ("born", "O"), ("in", "O"), ("Hawaii", "GPE")],
]


In [ ]:
def word2features(sent, i):
    word = sent[i][0]
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
    }
    if i > 0:
        word1 = sent[i-1][0]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
        })
    else:
        features['BOS'] = True  # Beginning of sentence
    if i < len(sent)-1:
        word1 = sent[i+1][0]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
        })
    else:
        features['EOS'] = True  # End of sentence
    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, label in sent]


In [ ]:
pip install sklearn-crfsuite


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.3 MB/s eta 0:00:00


In [ ]:
import sklearn_crfsuite


In [ ]:
import sklearn_crfsuite

X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    max_iterations=100,
    all_possible_transitions=True
)

crf.fit(X_train, y_train)


CRF(algorithm='lbfgs', all_possible_transitions=True, max_iterations=100)

In [ ]:
test_sent = [("Google", ""), ("is", ""), ("based", ""), ("in", ""), ("California", "")]
X_test = [sent2features(test_sent)]
y_pred = crf.predict(X_test)

for token, label in zip([w[0] for w in test_sent], y_pred[0]):
    print(token, label)


Google O
is O
based O
in O
California GPE


In [ ]:
# Remove consecutive duplicates text processing
text = "The the cat is is on the mat"

# Split into words
words = text.split()


new_words = [words[0]] + [words[i] for i in range(1, len(words)) if words[i] != words[i-1]]

clean_text = " ".join(new_words)
print(clean_text)


The the cat is on the mat
